In [1]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier



In [38]:
%pip install streamlit 

  Using cached altair-6.0.0-py3-none-any.whl.metadata (11 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached pandas-2.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached pydeck-0.9.1-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached pyarrow-23.0.1-cp312-cp312-macosx_12_0_arm64.whl.metadata (3.1 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached narwhals-2.18.0-py3-none-any.whl.metadata (14 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metad

In [2]:
!pip install pickle

ERROR: Could not find a version that satisfies the requirement pickle (from versions: none)

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for pickle


In [3]:
df=pd.read_csv('emi_cleaned.csv')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 392904 entries, 0 to 392903
Data columns (total 27 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   age                     392904 non-null  int64  
 1   gender                  392904 non-null  str    
 2   marital_status          392904 non-null  str    
 3   education               392904 non-null  str    
 4   monthly_salary          392904 non-null  int64  
 5   employment_type         392904 non-null  str    
 6   years_of_employment     392904 non-null  float64
 7   company_type            392904 non-null  str    
 8   house_type              392904 non-null  str    
 9   monthly_rent            392904 non-null  float64
 10  family_size             392904 non-null  int64  
 11  dependents              392904 non-null  int64  
 12  school_fees             392904 non-null  float64
 13  college_fees            392904 non-null  float64
 14  travel_expenses         392904 

In [5]:
gender_map={
    'M':'0',
    'F':'1',
}
df['gender']=df['gender'].replace(gender_map)
df['gender']=df['gender'].astype(int)

In [6]:
marital_status_map={
    'Single':'0',
    'Married':'1'
}
df['marital_status']=df['marital_status'].replace(marital_status_map)
df['marital_status']=df['marital_status'].astype(int)

In [7]:
education_map={
    'Unknown':'0',
    'High School':'1',
    'Post Graduate':'3',
    'Professional':'4',
    'Graduate':'2',
}
df['education']=df['education'].replace(education_map)
df['education']=df['education'].astype(int)

In [8]:
company_map={
    'Small':'0',
    'Startup':'1',
    'Mid-size':'2',
    'Large Indian':'3',
    'MNC':'4'
}
df['company_type']=df['company_type'].replace(company_map)
df['company_type']=df['company_type'].astype(int)

In [9]:
existing_loan_map={
    'No':'0',
    'Yes':'1'
}
df['existing_loans']=df['existing_loans'].replace(existing_loan_map)
df['existing_loans']=df['existing_loans'].astype(int)

'house_type'

In [10]:
cols_to_encode=['employment_type','emi_scenario',]#'emi_scenario',
df=pd.get_dummies(df,columns=cols_to_encode,prefix=cols_to_encode,dtype=int)

In [11]:
target_map={
    'Not_Eligible':'0',
    'High_Risk':'1',
    'Eligible':'2'
}
df['emi_eligibility']=df['emi_eligibility'].replace(target_map)
df['emi_eligibility']=df['emi_eligibility'].astype(int)

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 392904 entries, 0 to 392903
Data columns (total 33 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   age                                   392904 non-null  int64  
 1   gender                                392904 non-null  int64  
 2   marital_status                        392904 non-null  int64  
 3   education                             392904 non-null  int64  
 4   monthly_salary                        392904 non-null  int64  
 5   years_of_employment                   392904 non-null  float64
 6   company_type                          392904 non-null  int64  
 7   house_type                            392904 non-null  str    
 8   monthly_rent                          392904 non-null  float64
 9   family_size                           392904 non-null  int64  
 10  dependents                            392904 non-null  int64  
 11  school_fees

In [13]:
df['expense_salary_ratio']=(df['groceries_utilities']+df['college_fees']+df['current_emi_amount']+df['school_fees']+df['monthly_rent']+df['other_monthly_expenses']+df['travel_expenses'])/df['monthly_salary']
df=df.drop(columns=['house_type','groceries_utilities','college_fees','current_emi_amount','school_fees','monthly_rent','other_monthly_expenses','travel_expenses'])

In [14]:
X= df.drop(columns=['emi_eligibility','max_monthly_emi'])
y=df['emi_eligibility']

In [15]:
X.shape

(392904, 24)

In [16]:
X=X[['monthly_salary', 'existing_loans', 'credit_score', 'bank_balance',
       'emergency_fund', 'requested_amount', 'requested_tenure',
       'employment_type_Government', 'employment_type_Private',
       'emi_scenario_E-commerce Shopping EMI', 'emi_scenario_Education EMI',
       'emi_scenario_Home Appliances EMI', 'emi_scenario_Personal Loan EMI',
       'emi_scenario_Vehicle EMI', 'expense_salary_ratio']]

In [17]:
X.shape

(392904, 15)

In [18]:
st = StandardScaler()
X_scaled=st.fit_transform(X)

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [20]:
model = LogisticRegression()

In [21]:
model.fit(X_train,y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [22]:
pred=model.predict(X_test)

In [23]:
print("Accuracy:", accuracy_score(y_test,pred))
print(classification_report(y_test,pred))

Accuracy: 0.8873773558493784
              precision    recall  f1-score   support

           0       0.91      0.97      0.94     60744
           1       0.00      0.00      0.00      3353
           2       0.79      0.76      0.77     14484

    accuracy                           0.89     78581
   macro avg       0.57      0.57      0.57     78581
weighted avg       0.85      0.89      0.87     78581



In [24]:
from xgboost import XGBClassifier
xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=9,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

In [25]:
xgb.fit(X_train,y_train)
pred=xgb.predict(X_test)

In [26]:
print("Accuracy:", accuracy_score(y_test,pred))
print(classification_report(y_test,pred))

Accuracy: 0.9484480981407719
              precision    recall  f1-score   support

           0       0.96      1.00      0.98     60744
           1       0.47      0.01      0.02      3353
           2       0.91      0.97      0.94     14484

    accuracy                           0.95     78581
   macro avg       0.78      0.66      0.64     78581
weighted avg       0.93      0.95      0.93     78581



In [27]:
# rfe = RFE(estimator=xgb, n_features_to_select=15)

# X_train_rfe = rfe.fit_transform(X_train, y_train)
# X_test_rfe = rfe.transform(X_test)

# xgb.fit(X_train_rfe, y_train)

# y_pred = xgb.predict(X_test_rfe)

# selected_features = X.columns[rfe.support_]
# print("Selected Features:",selected_features)

In [28]:
# print("Accuracy:", accuracy_score(y_test,y_pred))
# print(classification_report(y_test,y_pred))

In [ ]:
rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
    min_samples_split=7,
    min_samples_leaf=5,
    max_depth=9,
    max_samples=0.7,
    n_estimators=150
)

In [30]:
pg={
    'n_estimators':[100,150],
    'max_depth':[5,7,9],
    'min_samples_split':[5,7],
    'min_samples_leaf':[3,5,7],
    'max_samples':[0.6,0.7]
}

In [31]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=rf, 
    param_distributions=pg, 
    n_iter=50,  # Only 50 combinations instead of 3072
    cv=3, 
    n_jobs=-1
)

In [32]:
random_search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'max_depth': [5, 7, ...], 'max_samples': [0.6, 0.7], 'min_samples_leaf': [3, 5, ...], 'min_samples_split': [5, 7], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchange

In [33]:
print(f"Best Parameters: {random_search.best_params_}")
print(f"Best Score (MSE): {random_search.best_score_}")

Best Parameters: {'n_estimators': 150, 'min_samples_split': 7, 'min_samples_leaf': 5, 'max_samples': 0.7, 'max_depth': 9}
Best Score (MSE): 0.9114605071093616


In [34]:
# # Train
# rf.fit(X_train, y_train)

# # Predict
# y_pred = rf.predict(X_test)

In [35]:
X_test.shape

(78581, 15)

In [36]:
X_test[0]

array([-0.15691924,  1.22728515, -0.99296253,  0.63672502, -0.08113459,
        1.60779872,  2.47922021,  1.99694777, -1.5248838 , -0.4998457 ,
       -0.49998091, -0.49990535, -0.5002036 ,  1.99974234,  0.61495291])

In [37]:
# print("Accuracy:", accuracy_score(y_test,y_pred))
# print(classification_report(y_test,y_pred))